In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

# =========================================================
# 0. CARGA Y NORMALIZACIÓN
# =========================================================
df    = pd.read_csv('../data/hr_employees_processed.csv')
df_jh = pd.read_csv('../data/hr_job_history_processed.csv')

# Normalizar nombres de columnas
df.columns    = df.columns.str.strip().str.lower()
df_jh.columns = df_jh.columns.str.strip().str.lower()

# Conversión de fechas (después de normalizar)
df['hiredate']      = pd.to_datetime(df['hiredate'])
df_jh['start_date'] = pd.to_datetime(df_jh['start_date'])
df_jh['end_date']   = pd.to_datetime(df_jh['end_date'])

#print("Columnas df:   ", df.columns.tolist())
#print("Columnas df_jh:", df_jh.columns.tolist())


# =========================================================================
# 1. HISTOGRAMA: Distribución de salarios por departamento
# =========================================================================
fig1 = px.histogram(
    df,
    x='salary',
    color='departmentname',
    nbins=30,
    barmode='overlay',
    title='1. Histograma: Distribución de Salarios por Departamento',
    labels={'salary': 'Salario ($)', 'departmentname': 'Departamento'},
    opacity=0.75
)
fig1.show()


# =========================================================================
# 2. GRÁFICO DE RADAR: KPIs por tipo de trabajo
# =========================================================================
top_jobs = df['jobtitle'].value_counts().head(6).index.tolist()
df_radar = df[df['jobtitle'].isin(top_jobs)]

radar_metrics = df_radar.groupby('jobtitle').agg(
    salario=('salary', 'mean'),
    antiguedad=('tenureyears', 'mean'),
    comision=('commissionpct', 'mean'),
    experiencia=('salary', lambda s: np.log1p(s).mean()),
    estabilidad=('tenureyears', lambda s: s.std())
).fillna(0)

radar_norm = (radar_metrics - radar_metrics.min()) / (radar_metrics.max() - radar_metrics.min() + 1e-9)

fig2 = go.Figure()

for job in radar_norm.index:
    fig2.add_trace(go.Scatterpolar(
        r=radar_norm.loc[job].values.tolist() + [radar_norm.loc[job].values[0]],
        theta=radar_norm.columns.tolist() + [radar_norm.columns[0]],
        fill='toself',
        name=job
    ))
fig2.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title='2. Gráfico de Radar: KPIs normalizados por Tipo de Trabajo',
    showlegend=True
)
fig2.show()


# =========================================================================
# 3. ORGANIGRAMA INTERACTIVO
# =========================================================================
org = df[['employeeid', 'employeename', 'managerid', 'managername',
          'departmentname', 'jobtitle', 'salary']].copy()
org['managername'] = org['managername'].fillna('CEO')

fig3 = px.treemap(
    org,
    path=['managername', 'employeename'],
    values='salary',
    color='departmentname',
    title='3. Organigrama Interactivo (Treemap)',
    hover_data=['jobtitle', 'departmentname', 'salary']
)
fig3.show()

fig3b = px.sunburst(
    org,
    path=['managername', 'employeename'],
    values='salary',
    color='departmentname',
    title='3b. Organigrama Radial (Sunburst)'
)
fig3b.show()


# =========================================================================
# 4. MAPA DE CALOR: Densidad de empleados por ubicación
# =========================================================================
loc_density = df.pivot_table(
    index='city',
    columns='departmentname',
    values='employeeid',
    aggfunc='count',
    fill_value=0
)

fig4 = px.imshow(
    loc_density,
    labels=dict(x="Departamento", y="Ciudad", color="Nº Empleados"),
    title='4. Mapa de Calor: Densidad de Empleados por Ubicación',
    color_continuous_scale='YlOrRd',
    text_auto=True,
    aspect='auto'
)
fig4.show()


# =========================================================================
# 5. GRÁFICO DE VIOLÍN: Análisis salarial por nivel de experiencia
# =========================================================================
df['experiencelevel'] = pd.cut(
    df['tenureyears'],
    bins=[-1, 2, 5, 10, 20, 100],
    labels=['0-2 años', '3-5 años', '6-10 años', '11-20 años', '20+ años']
)

fig5 = px.violin(
    df,
    x='experiencelevel',
    y='salary',
    color='experiencelevel',
    box=True,
    points='all',
    title='5. Gráfico de Violín: Análisis Salarial por Nivel de Experiencia',
    labels={'experiencelevel': 'Nivel de Experiencia', 'salary': 'Salario ($)'}
)
fig5.show()


# =========================================================================
# 6. SANKEY DIAGRAM: Flujo de cambios de departamento
# =========================================================================

jh = df_jh.merge(
    df[['employeeid', 'departmentname']].rename(columns={'departmentname': 'currentdept'}),
    left_on='employee_id',     # df_jh usa 'employee_id'
    right_on='employeeid',     # df usa 'employeeid'
    how='left'
)

print(f"Filas tras merge: {len(jh)}")
print(f"Filas con currentdept no nulo: {jh['currentdept'].notna().sum()}")

flows = jh.groupby(['department_name', 'currentdept']).size().reset_index(name='count')
flows = flows[flows['department_name'] != flows['currentdept']]

print(f"Flujos detectados: {len(flows)}")
print(flows.head())

if len(flows) > 0:
    all_nodes = pd.unique(flows[['department_name', 'currentdept']].values.ravel())
    node_idx = {n: i for i, n in enumerate(all_nodes)}

    fig6 = go.Figure(go.Sankey(
        node=dict(
            pad=20, thickness=20,
            line=dict(color='black', width=0.5),
            label=list(all_nodes),
            color='lightblue'
        ),
        link=dict(
            source=[node_idx[s] for s in flows['department_name']],
            target=[node_idx[t] for t in flows['currentdept']],
            value=flows['count'].tolist()
        )
    ))
    fig6.update_layout(title='6. Sankey Diagram: Flujo de Cambios de Departamento')
    fig6.show()
else:
    print("No hay flujos de cambio de departamento para graficar")

# =========================================================================
# 7. BUBBLE CHART: Salario vs. Antigüedad vs. Departamento
# =========================================================================
fig7 = px.scatter(
    df,
    x='tenureyears',
    y='salary',
    size='salary',
    color='departmentname',
    hover_data=['employeename', 'jobtitle'],
    title='7. Bubble Chart: Salario vs. Antigüedad vs. Departamento',
    labels={'tenureyears': 'Antigüedad (años)', 'salary': 'Salario ($)'},
    size_max=40,
    opacity=0.7
)
fig7.show()


# =========================================================================
# 8. GRÁFICO DE DENSIDAD: Concentración de contrataciones por mes
# =========================================================================
df['hiremonth'] = df['hiredate'].dt.month

hire_counts = df.groupby('hiremonth').size().reset_index(name='hires')

fig8 = go.Figure()
fig8.add_trace(go.Bar(
    x=hire_counts['hiremonth'],
    y=hire_counts['hires'],
    name='Contrataciones',
    marker_color='#636efa',
    opacity=0.5
))
fig8.add_trace(go.Scatter(
    x=hire_counts['hiremonth'],
    y=hire_counts['hires'],
    mode='lines+markers',
    name='Densidad (suavizado)',
    line=dict(color='#EF553B', width=3, shape='spline')
))
fig8.update_layout(
    title='8. Gráfico de Densidad: Concentración de Contrataciones por Mes',
    xaxis_title='Mes',
    yaxis_title='Nº de Contrataciones'
)
fig8.show()


# =========================================================================
# 9. DONUT CHART: Distribución de empleados por región
# =========================================================================
region_dist = df.groupby('regionname')['employeeid'].nunique().reset_index()
region_dist.columns = ['region', 'employeecount']

fig9 = px.pie(
    region_dist,
    names='region',
    values='employeecount',
    hole=0.55,
    title='9. Donut Chart: Distribución de Empleados por Región'
)
fig9.update_traces(textposition='inside', textinfo='percent+label')
fig9.show()


# =========================================================================
# 10. WATERFALL CHART: Progresión salarial en carrera
# =========================================================================
sample_emp = df_jh['employee_id'].value_counts().idxmax()
emp_hist = df_jh[df_jh['employee_id'] == sample_emp].sort_values('start_date')

sal_map = df.set_index('jobid')['salary'].to_dict()
emp_hist = emp_hist.assign(
    estimatedsalary=emp_hist['job_id'].map(sal_map).fillna(df['salary'].mean())
)

start_salary = emp_hist['estimatedsalary'].iloc[0]
deltas = emp_hist['estimatedsalary'].diff().fillna(0).tolist()

fig10 = go.Figure(go.Waterfall(
    name='Progresión salarial',
    orientation='v',
    measure=['absolute'] + ['relative'] * (len(deltas) - 1) + ['total'],
    x=['Inicio'] + [f'Cambio {i+1}' for i in range(len(deltas) - 1)] + ['Final'],
    y=[start_salary] + deltas[1:] + [0],
    connector=dict(line=dict(color='rgb(63, 63, 63)')),
    increasing=dict(marker=dict(color='#2ca02c')),
    decreasing=dict(marker=dict(color='#d62728')),
    totals=dict(marker=dict(color='#1f77b4'))
))
fig10.update_layout(
    title=f'10. Waterfall Chart: Progresión Salarial del Empleado {sample_emp}',
    yaxis_title='Salario ($)',
    xaxis_title='Etapa de Carrera'
)
fig10.show()

Filas tras merge: 16
Filas con currentdept no nulo: 16
Flujos detectados: 4
  department_name     currentdept  count
0      Accounting       Executive      4
1       Executive  Administration      4
2              IT       Executive      1
5        Shipping      Purchasing      1
